In [1]:
!pwd

/content


In [2]:
import sys;
print(sys.version)

3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [3]:
# !pip install uv

# !uv pip install \
#   "darts==0.34.0" \
#   "torch==2.3.1" \
#   "pytorch-lightning==2.2.5" \
#   "torchmetrics==1.3.2" \
#   "numpy==1.26.4" \
#   "pandas==2.2.2" \
#   "scikit-learn==1.7.2" \
#   "dask==2025.5.0" \
#   "xgboost==3.0.2" \
#   "catboost==1.2.8" \
#   "pyyaml==6.0.2" \
#   "lightgbm==4.6.0"

# Pin torch for reproducibility with your local run
# !uv pip install --index-url https://download.pytorch.org/whl/cu121 "torch==2.3.1"

In [4]:
import sys, torch, darts, pytorch_lightning as pl, numpy as np, pandas as pd
print(sys.version)
print("torch:", torch.__version__)
print("darts:", darts.__version__)
print("lightning:", pl.__version__)
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("cuda:", torch.cuda.is_available())

3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
torch: 2.3.1+cu121
darts: 0.34.0
lightning: 2.2.5
numpy: 1.26.4
pandas: 2.2.2
cuda: True


**Forecasts**
 - **Country: Canada**
 - **Forecast Horizons:12M and 24M Forecast**

In [9]:
import os
os.environ["PYTHONHASHSEED"] = "1"

import random
import pandas as pd
import numpy as np
import torch
from darts import TimeSeries
from darts.models import TFTModel
from typing import List, Tuple, Dict


class canadaForecastGenerator:
  def __init__(self, data_path: str, random_seed: int = 1):
      # Seeds
      random.seed(random_seed)
      torch.manual_seed(random_seed)
      np.random.seed(random_seed)

      self.data = pd.read_csv(data_path)

      self.target_vars = [
          "CPIinflationrate",
          "Unemploymentrate",
          "OilpriceGlobalWTI",
          "RealbroadEER",
          "ShorttermIR"
      ]

      self.exog_vars = [
          "logEPU",
          "GPRC",
          "USEMV",
          "USMPU"
      ]

  def prepare_data(self, forecast_horizon: int) -> Tuple[TimeSeries, TimeSeries]:
      train = self.data.head(-forecast_horizon).copy()

      target_series = [TimeSeries.from_series(train[var]) for var in self.target_vars]
      train_target_ts = target_series[0]
      for series in target_series[1:]:
          train_target_ts = train_target_ts.stack(series)

      exog_series = [TimeSeries.from_series(train[var]) for var in self.exog_vars]
      train_exog_ts = exog_series[0]
      for series in exog_series[1:]:
          train_exog_ts = train_exog_ts.stack(series)

      return train_target_ts, train_exog_ts

  def create_model(self, forecast_horizon: int) -> TFTModel:
      # HPs unchanged; only runtime forced to CPU for reproducibility
      return TFTModel(
          input_chunk_length=3,
          output_chunk_length=forecast_horizon,
          hidden_size=6,
          lstm_layers=3,
          num_attention_heads=3,
          full_attention=True,
          dropout=0.002,
          hidden_continuous_size=4,
          n_epochs=300,
          random_state=0,
          add_relative_index=True,
          pl_trainer_kwargs={
              "accelerator": "cpu",
              "devices": 1,
              "enable_checkpointing": False,
              "logger": False,
          }
      )

  def generate_forecasts(self, forecast_horizons: List[int]) -> Dict[int, pd.DataFrame]:
      forecasts = {}

      for horizon in forecast_horizons:
          print(f"\nGenerating {horizon}-month forecast...")

          train_target_ts, train_exog_ts = self.prepare_data(horizon)

          model = self.create_model(horizon)
          print(f"Training model for {horizon}-month horizon...")
          model.fit(
              series=train_target_ts,
              past_covariates=train_exog_ts,
              verbose=True
          )

          print(f"Generating {horizon}-month predictions...")
          pred = model.predict(
              n=horizon,
              series=train_target_ts,
              past_covariates=train_exog_ts
          )

          pred_df = pd.DataFrame({
              "forecast_inflation": pred.pd_dataframe().iloc[:, 0],
              "forecast_unemployment": pred.pd_dataframe().iloc[:, 1],
              "forecast_oil_price": pred.pd_dataframe().iloc[:, 2],
              "forecast_eer": pred.pd_dataframe().iloc[:, 3],
              "forecast_ir": pred.pd_dataframe().iloc[:, 4]
          })

          forecasts[horizon] = pred_df

          output_file = f"canada_forecasts_tftx_{horizon}m.csv"
          pred_df.to_csv(output_file, index=True)
          print(f"Forecast saved to {output_file}")

      return forecasts


def main():
  import sys, darts, pytorch_lightning as pl
  print(sys.version)
  print("torch:", torch.__version__)
  print("darts:", darts.__version__)
  print("lightning:", pl.__version__)
  print("numpy:", np.__version__)
  print("pandas:", pd.__version__)
  print("cuda available:", torch.cuda.is_available())

  generator = canadaForecastGenerator(
      data_path="all_mulvar_data_canada_v2.csv",
      random_seed=1
  )

  generator.generate_forecasts([12, 24])

  print("\nCreated files:")
  print("canada_forecasts_tftx_12m.csv")
  print("canada_forecasts_tftx_24m.csv")


if __name__ == "__main__":
  main()

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/setup.py:187: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
INFO:pytorch_lightning.callbacks.model_summary:
   | Name                              | Type                             | Params
----------------------------------------------------------------------------------------
0  | train_metrics                     | MetricCollection                 | 0     
1  | val_metrics                       | MetricCollection                 | 0     
2  | input_embeddings                  | _MultiEmbedding                  | 0     
3  | static_covariates_vsn             | _Va

3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
torch: 2.3.1+cu121
darts: 0.34.0
lightning: 2.2.5
numpy: 1.26.4
pandas: 2.2.2
cuda available: True

Generating 12-month forecast...
Training model for 12-month horizon...


Training: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=300` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


Generating 12-month predictions...


...

Training: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=300` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


Generating 24-month predictions...


Predicting: |          | 0/? [00:00<?, ?it/s]

Forecast saved to canada_forecasts_tftx_24m.csv

Created files:
canada_forecasts_tftx_12m.csv
canada_forecasts_tftx_24m.csv


**Forecasts**
 - **Country: USA**
 - **Forecast Horizons:12M and 24M Forecast**

In [11]:
import os
import random
import pandas as pd
import numpy as np
import torch
import pytorch_lightning as pl

from darts import TimeSeries
from darts.models import TFTModel
from typing import List, Tuple, Dict

# ---------------------------
# Reproducibility controls
# ---------------------------
os.environ["PYTHONHASHSEED"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"

random.seed(1)
np.random.seed(1)
torch.manual_seed(1)
pl.seed_everything(1, workers=True)

# Safe thread settings for Colab/Jupyter
try:
    torch.set_num_threads(1)
except Exception:
    pass

try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    # Happens if parallel backend already initialized in this kernel
    pass

try:
    torch.use_deterministic_algorithms(True)
except Exception:
    pass


class usaForecastGenerator:
    def __init__(self, data_path: str, random_seed: int = 1):
        random.seed(random_seed)
        torch.manual_seed(random_seed)
        np.random.seed(random_seed)
        pl.seed_everything(random_seed, workers=True)

        self.data = pd.read_csv(data_path)

        self.target_vars = [
            "CPIinflationrate",
            "Unemploymentrate",
            "OilpriceGlobalWTI",
            "RealbroadEER",
            "ShorttermIR"
        ]

        self.exog_vars = [
            "logEPU",
            "GPRC",
            "USEMV",
            "USMPU"
        ]

    def prepare_data(self, forecast_horizon: int) -> Tuple[TimeSeries, TimeSeries]:
        train = self.data.head(-forecast_horizon).copy()

        target_series = [TimeSeries.from_series(train[var]) for var in self.target_vars]
        train_target_ts = target_series[0]
        for series in target_series[1:]:
            train_target_ts = train_target_ts.stack(series)

        exog_series = [TimeSeries.from_series(train[var]) for var in self.exog_vars]
        train_exog_ts = exog_series[0]
        for series in exog_series[1:]:
            train_exog_ts = train_exog_ts.stack(series)

        return train_target_ts, train_exog_ts

    def create_model(self, forecast_horizon: int) -> TFTModel:
        # Model structure + HPs unchanged
        return TFTModel(
            input_chunk_length=3,
            output_chunk_length=forecast_horizon,
            hidden_size=6,
            lstm_layers=3,
            num_attention_heads=3,
            full_attention=True,
            dropout=0.002,
            hidden_continuous_size=4,
            n_epochs=300,
            random_state=0,
            add_relative_index=True,
            pl_trainer_kwargs={
                "accelerator": "cpu",
                "devices": 1,
                "logger": False,
                "enable_checkpointing": False,
            }
        )

    def generate_forecasts(self, forecast_horizons: List[int]) -> Dict[int, pd.DataFrame]:
        forecasts = {}

        for horizon in forecast_horizons:
            print(f"\nGenerating {horizon}-month forecast...")

            train_target_ts, train_exog_ts = self.prepare_data(horizon)

            model = self.create_model(horizon)
            print(f"Training model for {horizon}-month horizon...")
            model.fit(
                series=train_target_ts,
                past_covariates=train_exog_ts,
                verbose=True
            )

            print(f"Generating {horizon}-month predictions...")
            pred = model.predict(
                n=horizon,
                series=train_target_ts,
                past_covariates=train_exog_ts
            )

            pred_df = pd.DataFrame({
                "forecast_inflation": pred.pd_dataframe().iloc[:, 0],
                "forecast_unemployment": pred.pd_dataframe().iloc[:, 1],
                "forecast_oil_price": pred.pd_dataframe().iloc[:, 2],
                "forecast_eer": pred.pd_dataframe().iloc[:, 3],
                "forecast_ir": pred.pd_dataframe().iloc[:, 4]
            })

            forecasts[horizon] = pred_df

            output_file = f"usa_forecasts_tftx_{horizon}m.csv"
            pred_df.to_csv(output_file, index=True)
            print(f"Forecast saved to {output_file}")

        return forecasts


def main():
    import sys, darts
    print(sys.version)
    print("torch:", torch.__version__)
    print("darts:", darts.__version__)
    print("lightning:", pl.__version__)
    print("numpy:", np.__version__)
    print("pandas:", pd.__version__)
    print("cuda available:", torch.cuda.is_available())

    generator = usaForecastGenerator(
        data_path="/content/all_mulvar_data_usa_v2.csv",
        random_seed=1
    )

    generator.generate_forecasts([12, 24])

    print("\nCreated files:")
    print("usa_forecasts_tftx_12m.csv")
    print("usa_forecasts_tftx_24m.csv")


if __name__ == "__main__":
    main()

INFO:lightning_fabric.utilities.seed:Seed set to 1
INFO:lightning_fabric.utilities.seed:Seed set to 1
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/setup.py:187: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
INFO:pytorch_lightning.callbacks.model_summary:
   | Name                              | Type                             | Params
----------------------------------------------------------------------------------------
0  | train_metrics                     | MetricCollection                 | 0     
1  | val_metrics                       | MetricCollection                 | 0     
2  | input_embeddings    

3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
torch: 2.3.1+cu121
darts: 0.34.0
lightning: 2.2.5
numpy: 1.26.4
pandas: 2.2.2
cuda available: True

Generating 12-month forecast...
Training model for 12-month horizon...


Training: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=300` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


Generating 12-month predictions...


...

Training: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=300` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


Generating 24-month predictions...


Predicting: |          | 0/? [00:00<?, ?it/s]

Forecast saved to usa_forecasts_tftx_24m.csv

Created files:
usa_forecasts_tftx_12m.csv
usa_forecasts_tftx_24m.csv


**Forecasts**
 - **Country: France**
 - **Forecast Horizons:12M and 24M Forecast**

In [12]:
import os
import random
import pandas as pd
import numpy as np
import torch
import pytorch_lightning as pl

from darts import TimeSeries
from darts.models import TFTModel
from typing import List, Tuple, Dict

# ---------------------------
# Reproducibility controls
# ---------------------------
os.environ["PYTHONHASHSEED"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"

random.seed(1)
np.random.seed(1)
torch.manual_seed(1)
pl.seed_everything(1, workers=True)

# Safe for Colab/Jupyter
try:
    torch.set_num_threads(1)
except Exception:
    pass

try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass

try:
    torch.use_deterministic_algorithms(True)
except Exception:
    pass


class franceForecastGenerator:
    def __init__(self, data_path: str, random_seed: int = 1):
        random.seed(random_seed)
        torch.manual_seed(random_seed)
        np.random.seed(random_seed)
        pl.seed_everything(random_seed, workers=True)

        self.data = pd.read_csv(data_path)

        self.target_vars = [
            "CPIinflationrate",
            "Unemploymentrate",
            "OilpriceGlobalWTI",
            "RealbroadEER",
            "ShorttermIR"
        ]

        self.exog_vars = [
            "logEPU",
            "GPRC",
            "USEMV",
            "USMPU"
        ]

    def prepare_data(self, forecast_horizon: int) -> Tuple[TimeSeries, TimeSeries]:
        train = self.data.head(-forecast_horizon).copy()

        target_series = [TimeSeries.from_series(train[var]) for var in self.target_vars]
        train_target_ts = target_series[0]
        for series in target_series[1:]:
            train_target_ts = train_target_ts.stack(series)

        exog_series = [TimeSeries.from_series(train[var]) for var in self.exog_vars]
        train_exog_ts = exog_series[0]
        for series in exog_series[1:]:
            train_exog_ts = train_exog_ts.stack(series)

        return train_target_ts, train_exog_ts

    def create_model(self, forecast_horizon: int) -> TFTModel:
        # Model structure + HPs unchanged
        return TFTModel(
            input_chunk_length=3,
            output_chunk_length=forecast_horizon,
            hidden_size=6,
            lstm_layers=3,
            num_attention_heads=3,
            full_attention=True,
            dropout=0.002,
            hidden_continuous_size=4,
            n_epochs=300,
            random_state=0,
            add_relative_index=True,
            pl_trainer_kwargs={
                "accelerator": "cpu",
                "devices": 1,
                "logger": False,
                "enable_checkpointing": False,
            }
        )

    def generate_forecasts(self, forecast_horizons: List[int]) -> Dict[int, pd.DataFrame]:
        forecasts = {}

        for horizon in forecast_horizons:
            print(f"\nGenerating {horizon}-month forecast...")

            train_target_ts, train_exog_ts = self.prepare_data(horizon)

            model = self.create_model(horizon)
            print(f"Training model for {horizon}-month horizon...")
            model.fit(
                series=train_target_ts,
                past_covariates=train_exog_ts,
                verbose=True
            )

            print(f"Generating {horizon}-month predictions...")
            pred = model.predict(
                n=horizon,
                series=train_target_ts,
                past_covariates=train_exog_ts
            )

            pred_df = pd.DataFrame({
                "forecast_inflation": pred.pd_dataframe().iloc[:, 0],
                "forecast_unemployment": pred.pd_dataframe().iloc[:, 1],
                "forecast_oil_price": pred.pd_dataframe().iloc[:, 2],
                "forecast_eer": pred.pd_dataframe().iloc[:, 3],
                "forecast_ir": pred.pd_dataframe().iloc[:, 4]
            })

            forecasts[horizon] = pred_df

            output_file = f"france_forecasts_tftx_{horizon}m.csv"
            pred_df.to_csv(output_file, index=True)
            print(f"Forecast saved to {output_file}")

        return forecasts


def main():
    import sys, darts
    print(sys.version)
    print("torch:", torch.__version__)
    print("darts:", darts.__version__)
    print("lightning:", pl.__version__)
    print("numpy:", np.__version__)
    print("pandas:", pd.__version__)
    print("cuda available:", torch.cuda.is_available())

    generator = franceForecastGenerator(
        data_path="/content/all_mulvar_data_france_v2.csv",
        random_seed=1
    )

    generator.generate_forecasts([12, 24])

    print("\nCreated files:")
    print("france_forecasts_tftx_12m.csv")
    print("france_forecasts_tftx_24m.csv")


if __name__ == "__main__":
    main()

INFO:lightning_fabric.utilities.seed:Seed set to 1
INFO:lightning_fabric.utilities.seed:Seed set to 1


3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
torch: 2.3.1+cu121
darts: 0.34.0
lightning: 2.2.5
numpy: 1.26.4
pandas: 2.2.2
cuda available: True

Generating 12-month forecast...
Training model for 12-month horizon...


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/setup.py:187: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
INFO:pytorch_lightning.callbacks.model_summary:
   | Name                              | Type                             | Params
----------------------------------------------------------------------------------------
0  | train_metrics                     | MetricCollection                 | 0     
1  | val_metrics                       | MetricCollection                 | 0     
2  | input_embeddings                  | _MultiEmbedding                  | 0     
3  | static_covariates_vsn             | _Va

Training: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=300` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


...

Training: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=300` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


Generating 24-month predictions...


Predicting: |          | 0/? [00:00<?, ?it/s]

Forecast saved to france_forecasts_tftx_24m.csv

Created files:
france_forecasts_tftx_12m.csv
france_forecasts_tftx_24m.csv


**Forecasts**
 - **Country: Germany**
 - **Forecast Horizons:12M and 24M Forecast**

In [13]:
import os
import random
import pandas as pd
import numpy as np
import torch
import pytorch_lightning as pl

from darts import TimeSeries
from darts.models import TFTModel
from typing import List, Tuple, Dict

# ---------------------------
# Reproducibility controls
# ---------------------------
os.environ["PYTHONHASHSEED"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"

random.seed(1)
np.random.seed(1)
torch.manual_seed(1)
pl.seed_everything(1, workers=True)

# Safe for Colab/Jupyter
try:
    torch.set_num_threads(1)
except Exception:
    pass

try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass

try:
    torch.use_deterministic_algorithms(True)
except Exception:
    pass


class germanyForecastGenerator:
    def __init__(self, data_path: str, random_seed: int = 1):
        random.seed(random_seed)
        torch.manual_seed(random_seed)
        np.random.seed(random_seed)
        pl.seed_everything(random_seed, workers=True)

        self.data = pd.read_csv(data_path)

        self.target_vars = [
            "CPIinflationrate",
            "Unemploymentrate",
            "OilpriceGlobalWTI",
            "RealbroadEER",
            "ShorttermIR"
        ]

        self.exog_vars = [
            "logEPU",
            "GPRC",
            "USEMV",
            "USMPU"
        ]

    def prepare_data(self, forecast_horizon: int) -> Tuple[TimeSeries, TimeSeries]:
        train = self.data.head(-forecast_horizon).copy()

        target_series = [TimeSeries.from_series(train[var]) for var in self.target_vars]
        train_target_ts = target_series[0]
        for series in target_series[1:]:
            train_target_ts = train_target_ts.stack(series)

        exog_series = [TimeSeries.from_series(train[var]) for var in self.exog_vars]
        train_exog_ts = exog_series[0]
        for series in exog_series[1:]:
            train_exog_ts = train_exog_ts.stack(series)

        return train_target_ts, train_exog_ts

    def create_model(self, forecast_horizon: int) -> TFTModel:
        # Model structure + HPs unchanged
        return TFTModel(
            input_chunk_length=3,
            output_chunk_length=forecast_horizon,
            hidden_size=6,
            lstm_layers=3,
            num_attention_heads=3,
            full_attention=True,
            dropout=0.002,
            hidden_continuous_size=4,
            n_epochs=300,
            random_state=0,
            add_relative_index=True,
            pl_trainer_kwargs={
                "accelerator": "cpu",
                "devices": 1,
                "logger": False,
                "enable_checkpointing": False,
            }
        )

    def generate_forecasts(self, forecast_horizons: List[int]) -> Dict[int, pd.DataFrame]:
        forecasts = {}

        for horizon in forecast_horizons:
            print(f"\nGenerating {horizon}-month forecast...")

            train_target_ts, train_exog_ts = self.prepare_data(horizon)

            model = self.create_model(horizon)
            print(f"Training model for {horizon}-month horizon...")
            model.fit(
                series=train_target_ts,
                past_covariates=train_exog_ts,
                verbose=True
            )

            print(f"Generating {horizon}-month predictions...")
            pred = model.predict(
                n=horizon,
                series=train_target_ts,
                past_covariates=train_exog_ts
            )

            pred_df = pd.DataFrame({
                "forecast_inflation": pred.pd_dataframe().iloc[:, 0],
                "forecast_unemployment": pred.pd_dataframe().iloc[:, 1],
                "forecast_oil_price": pred.pd_dataframe().iloc[:, 2],
                "forecast_eer": pred.pd_dataframe().iloc[:, 3],
                "forecast_ir": pred.pd_dataframe().iloc[:, 4]
            })

            forecasts[horizon] = pred_df

            output_file = f"germany_forecasts_tftx_{horizon}m.csv"
            pred_df.to_csv(output_file, index=True)
            print(f"Forecast saved to {output_file}")

        return forecasts


def main():
    import sys, darts
    print(sys.version)
    print("torch:", torch.__version__)
    print("darts:", darts.__version__)
    print("lightning:", pl.__version__)
    print("numpy:", np.__version__)
    print("pandas:", pd.__version__)
    print("cuda available:", torch.cuda.is_available())

    generator = germanyForecastGenerator(
        data_path="/content/all_mulvar_data_germany_v2.csv",
        random_seed=1
    )

    generator.generate_forecasts([12, 24])

    print("\nCreated files:")
    print("germany_forecasts_tftx_12m.csv")
    print("germany_forecasts_tftx_24m.csv")


if __name__ == "__main__":
    main()

INFO:lightning_fabric.utilities.seed:Seed set to 1
INFO:lightning_fabric.utilities.seed:Seed set to 1


3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
torch: 2.3.1+cu121
darts: 0.34.0
lightning: 2.2.5
numpy: 1.26.4
pandas: 2.2.2
cuda available: True

Generating 12-month forecast...


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/setup.py:187: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
INFO:pytorch_lightning.callbacks.model_summary:
   | Name                              | Type                             | Params
----------------------------------------------------------------------------------------
0  | train_metrics                     | MetricCollection                 | 0     
1  | val_metrics                       | MetricCollection                 | 0     
2  | input_embeddings                  | _MultiEmbedding                  | 0     
3  | static_covariates_vsn             | _Va

Training model for 12-month horizon...


Training: |          | 0/? [00:00<?, ?it/s]

...

Training: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=300` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


Generating 24-month predictions...


Predicting: |          | 0/? [00:00<?, ?it/s]

Forecast saved to germany_forecasts_tftx_24m.csv

Created files:
germany_forecasts_tftx_12m.csv
germany_forecasts_tftx_24m.csv


**Forecasts**
 - **Country: Japan**
 - **Forecast Horizons:12M and 24M Forecast**

In [14]:
import os
import random
import pandas as pd
import numpy as np
import torch
import pytorch_lightning as pl

from darts import TimeSeries
from darts.models import TFTModel
from typing import List, Tuple, Dict

# ---------------------------
# Reproducibility controls
# ---------------------------
os.environ["PYTHONHASHSEED"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"

random.seed(1)
np.random.seed(1)
torch.manual_seed(1)
pl.seed_everything(1, workers=True)

# Safe for Colab/Jupyter
try:
    torch.set_num_threads(1)
except Exception:
    pass

try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass

try:
    torch.use_deterministic_algorithms(True)
except Exception:
    pass


class japanForecastGenerator:
    def __init__(self, data_path: str, random_seed: int = 1):
        random.seed(random_seed)
        torch.manual_seed(random_seed)
        np.random.seed(random_seed)
        pl.seed_everything(random_seed, workers=True)

        self.data = pd.read_csv(data_path)

        self.target_vars = [
            "CPIinflationrate",
            "Unemploymentrate",
            "OilpriceGlobalWTI",
            "RealbroadEER",
            "ShorttermIR"
        ]

        self.exog_vars = [
            "logEPU",
            "GPRC",
            "USEMV",
            "USMPU"
        ]

    def prepare_data(self, forecast_horizon: int) -> Tuple[TimeSeries, TimeSeries]:
        train = self.data.head(-forecast_horizon).copy()

        target_series = [TimeSeries.from_series(train[var]) for var in self.target_vars]
        train_target_ts = target_series[0]
        for series in target_series[1:]:
            train_target_ts = train_target_ts.stack(series)

        exog_series = [TimeSeries.from_series(train[var]) for var in self.exog_vars]
        train_exog_ts = exog_series[0]
        for series in exog_series[1:]:
            train_exog_ts = train_exog_ts.stack(series)

        return train_target_ts, train_exog_ts

    def create_model(self, forecast_horizon: int) -> TFTModel:
        # Model structure + HPs unchanged
        return TFTModel(
            input_chunk_length=3,
            output_chunk_length=forecast_horizon,
            hidden_size=6,
            lstm_layers=3,
            num_attention_heads=3,
            full_attention=True,
            dropout=0.002,
            hidden_continuous_size=4,
            n_epochs=300,
            random_state=0,
            add_relative_index=True,
            pl_trainer_kwargs={
                "accelerator": "cpu",
                "devices": 1,
                "logger": False,
                "enable_checkpointing": False,
            }
        )

    def generate_forecasts(self, forecast_horizons: List[int]) -> Dict[int, pd.DataFrame]:
        forecasts = {}

        for horizon in forecast_horizons:
            print(f"\nGenerating {horizon}-month forecast...")

            train_target_ts, train_exog_ts = self.prepare_data(horizon)

            model = self.create_model(horizon)
            print(f"Training model for {horizon}-month horizon...")
            model.fit(
                series=train_target_ts,
                past_covariates=train_exog_ts,
                verbose=True
            )

            print(f"Generating {horizon}-month predictions...")
            pred = model.predict(
                n=horizon,
                series=train_target_ts,
                past_covariates=train_exog_ts
            )

            pred_df = pd.DataFrame({
                "forecast_inflation": pred.pd_dataframe().iloc[:, 0],
                "forecast_unemployment": pred.pd_dataframe().iloc[:, 1],
                "forecast_oil_price": pred.pd_dataframe().iloc[:, 2],
                "forecast_eer": pred.pd_dataframe().iloc[:, 3],
                "forecast_ir": pred.pd_dataframe().iloc[:, 4]
            })

            forecasts[horizon] = pred_df

            output_file = f"japan_forecasts_tftx_{horizon}m.csv"
            pred_df.to_csv(output_file, index=True)
            print(f"Forecast saved to {output_file}")

        return forecasts


def main():
    import sys, darts
    print(sys.version)
    print("torch:", torch.__version__)
    print("darts:", darts.__version__)
    print("lightning:", pl.__version__)
    print("numpy:", np.__version__)
    print("pandas:", pd.__version__)
    print("cuda available:", torch.cuda.is_available())

    generator = japanForecastGenerator(
        data_path="/content/all_mulvar_data_japan_v2.csv",
        random_seed=1
    )

    generator.generate_forecasts([12, 24])

    print("\nCreated files:")
    print("japan_forecasts_tftx_12m.csv")
    print("japan_forecasts_tftx_24m.csv")


if __name__ == "__main__":
    main()

INFO:lightning_fabric.utilities.seed:Seed set to 1
INFO:lightning_fabric.utilities.seed:Seed set to 1


3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
torch: 2.3.1+cu121
darts: 0.34.0
lightning: 2.2.5
numpy: 1.26.4
pandas: 2.2.2
cuda available: True

Generating 12-month forecast...
Training model for 12-month horizon...


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/setup.py:187: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
INFO:pytorch_lightning.callbacks.model_summary:
   | Name                              | Type                             | Params
----------------------------------------------------------------------------------------
0  | train_metrics                     | MetricCollection                 | 0     
1  | val_metrics                       | MetricCollection                 | 0     
2  | input_embeddings                  | _MultiEmbedding                  | 0     
3  | static_covariates_vsn             | _Va

Training: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=300` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


...

Training: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=300` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


Generating 24-month predictions...


Predicting: |          | 0/? [00:00<?, ?it/s]

Forecast saved to japan_forecasts_tftx_24m.csv

Created files:
japan_forecasts_tftx_12m.csv
japan_forecasts_tftx_24m.csv


**Forecasts**
 - **Country: UK**
 - **Forecast Horizons:12M and 24M Forecast**

In [15]:
import os
import random
import pandas as pd
import numpy as np
import torch
import pytorch_lightning as pl

from darts import TimeSeries
from darts.models import TFTModel
from typing import List, Tuple, Dict

# ---------------------------
# Reproducibility controls
# ---------------------------
os.environ["PYTHONHASHSEED"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"

random.seed(1)
np.random.seed(1)
torch.manual_seed(1)
pl.seed_everything(1, workers=True)

# Safe for Colab/Jupyter
try:
    torch.set_num_threads(1)
except Exception:
    pass

try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass

try:
    torch.use_deterministic_algorithms(True)
except Exception:
    pass


class ukForecastGenerator:
    def __init__(self, data_path: str, random_seed: int = 1):
        random.seed(random_seed)
        torch.manual_seed(random_seed)
        np.random.seed(random_seed)
        pl.seed_everything(random_seed, workers=True)

        self.data = pd.read_csv(data_path)

        self.target_vars = [
            "CPIinflationrate",
            "Unemploymentrate",
            "OilpriceGlobalWTI",
            "RealbroadEER",
            "ShorttermIR"
        ]

        self.exog_vars = [
            "logEPU",
            "GPRC",
            "USEMV",
            "USMPU"
        ]

    def prepare_data(self, forecast_horizon: int) -> Tuple[TimeSeries, TimeSeries]:
        train = self.data.head(-forecast_horizon).copy()

        target_series = [TimeSeries.from_series(train[var]) for var in self.target_vars]
        train_target_ts = target_series[0]
        for series in target_series[1:]:
            train_target_ts = train_target_ts.stack(series)

        exog_series = [TimeSeries.from_series(train[var]) for var in self.exog_vars]
        train_exog_ts = exog_series[0]
        for series in exog_series[1:]:
            train_exog_ts = train_exog_ts.stack(series)

        return train_target_ts, train_exog_ts

    def create_model(self, forecast_horizon: int) -> TFTModel:
        # Model structure + HPs unchanged
        return TFTModel(
            input_chunk_length=3,
            output_chunk_length=forecast_horizon,
            hidden_size=6,
            lstm_layers=3,
            num_attention_heads=3,
            full_attention=True,
            dropout=0.002,
            hidden_continuous_size=4,
            n_epochs=300,
            random_state=0,
            add_relative_index=True,
            pl_trainer_kwargs={
                "accelerator": "cpu",
                "devices": 1,
                "logger": False,
                "enable_checkpointing": False,
            }
        )

    def generate_forecasts(self, forecast_horizons: List[int]) -> Dict[int, pd.DataFrame]:
        forecasts = {}

        for horizon in forecast_horizons:
            print(f"\nGenerating {horizon}-month forecast...")

            train_target_ts, train_exog_ts = self.prepare_data(horizon)

            model = self.create_model(horizon)
            print(f"Training model for {horizon}-month horizon...")
            model.fit(
                series=train_target_ts,
                past_covariates=train_exog_ts,
                verbose=True
            )

            print(f"Generating {horizon}-month predictions...")
            pred = model.predict(
                n=horizon,
                series=train_target_ts,
                past_covariates=train_exog_ts
            )

            pred_df = pd.DataFrame({
                "forecast_inflation": pred.pd_dataframe().iloc[:, 0],
                "forecast_unemployment": pred.pd_dataframe().iloc[:, 1],
                "forecast_oil_price": pred.pd_dataframe().iloc[:, 2],
                "forecast_eer": pred.pd_dataframe().iloc[:, 3],
                "forecast_ir": pred.pd_dataframe().iloc[:, 4]
            })

            forecasts[horizon] = pred_df

            output_file = f"uk_forecasts_tftx_{horizon}m.csv"
            pred_df.to_csv(output_file, index=True)
            print(f"Forecast saved to {output_file}")

        return forecasts


def main():
    import sys, darts
    print(sys.version)
    print("torch:", torch.__version__)
    print("darts:", darts.__version__)
    print("lightning:", pl.__version__)
    print("numpy:", np.__version__)
    print("pandas:", pd.__version__)
    print("cuda available:", torch.cuda.is_available())

    generator = ukForecastGenerator(
        data_path="/content/all_mulvar_data_uk_v2.csv",
        random_seed=1
    )

    generator.generate_forecasts([12, 24])

    print("\nCreated files:")
    print("uk_forecasts_tftx_12m.csv")
    print("uk_forecasts_tftx_24m.csv")


if __name__ == "__main__":
    main()

INFO:lightning_fabric.utilities.seed:Seed set to 1
INFO:lightning_fabric.utilities.seed:Seed set to 1


3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
torch: 2.3.1+cu121
darts: 0.34.0
lightning: 2.2.5
numpy: 1.26.4
pandas: 2.2.2
cuda available: True

Generating 12-month forecast...
Training model for 12-month horizon...


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/setup.py:187: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
INFO:pytorch_lightning.callbacks.model_summary:
   | Name                              | Type                             | Params
----------------------------------------------------------------------------------------
0  | train_metrics                     | MetricCollection                 | 0     
1  | val_metrics                       | MetricCollection                 | 0     
2  | input_embeddings                  | _MultiEmbedding                  | 0     
3  | static_covariates_vsn             | _Va

Training: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=300` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


...

Training: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=300` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


Generating 24-month predictions...


Predicting: |          | 0/? [00:00<?, ?it/s]

Forecast saved to uk_forecasts_tftx_24m.csv

Created files:
uk_forecasts_tftx_12m.csv
uk_forecasts_tftx_24m.csv


**Forecasts**
 - **Country: Italy**
 - **Forecast Horizons:12M and 24M Forecast**


In [16]:
import os
import random
import pandas as pd
import numpy as np
import torch
import pytorch_lightning as pl

from darts import TimeSeries
from darts.models import TFTModel
from typing import List, Tuple, Dict

# ---------------------------
# Reproducibility controls
# ---------------------------
os.environ["PYTHONHASHSEED"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"

random.seed(1)
np.random.seed(1)
torch.manual_seed(1)
pl.seed_everything(1, workers=True)

# Safe for Colab/Jupyter
try:
    torch.set_num_threads(1)
except Exception:
    pass

try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass

try:
    torch.use_deterministic_algorithms(True)
except Exception:
    pass


class italyForecastGenerator:
    def __init__(self, data_path: str, random_seed: int = 1):
        random.seed(random_seed)
        torch.manual_seed(random_seed)
        np.random.seed(random_seed)
        pl.seed_everything(random_seed, workers=True)

        self.data = pd.read_csv(data_path)

        self.target_vars = [
            "CPIinflationrate",
            "Unemploymentrate",
            "OilpriceGlobalWTI",
            "RealbroadEER",
            "ShorttermIR"
        ]

        self.exog_vars = [
            "logEPU",
            "GPRC",
            "USEMV",
            "USMPU"
        ]

    def prepare_data(self, forecast_horizon: int) -> Tuple[TimeSeries, TimeSeries]:
        train = self.data.head(-forecast_horizon).copy()

        target_series = [TimeSeries.from_series(train[var]) for var in self.target_vars]
        train_target_ts = target_series[0]
        for series in target_series[1:]:
            train_target_ts = train_target_ts.stack(series)

        exog_series = [TimeSeries.from_series(train[var]) for var in self.exog_vars]
        train_exog_ts = exog_series[0]
        for series in exog_series[1:]:
            train_exog_ts = train_exog_ts.stack(series)

        return train_target_ts, train_exog_ts

    def create_model(self, forecast_horizon: int) -> TFTModel:
        # Model structure + HPs unchanged
        return TFTModel(
            input_chunk_length=3,
            output_chunk_length=forecast_horizon,
            hidden_size=6,
            lstm_layers=3,
            num_attention_heads=3,
            full_attention=True,
            dropout=0.002,
            hidden_continuous_size=4,
            n_epochs=300,
            random_state=0,
            add_relative_index=True,
            pl_trainer_kwargs={
                "accelerator": "cpu",
                "devices": 1,
                "logger": False,
                "enable_checkpointing": False,
            }
        )

    def generate_forecasts(self, forecast_horizons: List[int]) -> Dict[int, pd.DataFrame]:
        forecasts = {}

        for horizon in forecast_horizons:
            print(f"\nGenerating {horizon}-month forecast...")

            train_target_ts, train_exog_ts = self.prepare_data(horizon)

            model = self.create_model(horizon)
            print(f"Training model for {horizon}-month horizon...")
            model.fit(
                series=train_target_ts,
                past_covariates=train_exog_ts,
                verbose=True
            )

            print(f"Generating {horizon}-month predictions...")
            pred = model.predict(
                n=horizon,
                series=train_target_ts,
                past_covariates=train_exog_ts
            )

            pred_df = pd.DataFrame({
                "forecast_inflation": pred.pd_dataframe().iloc[:, 0],
                "forecast_unemployment": pred.pd_dataframe().iloc[:, 1],
                "forecast_oil_price": pred.pd_dataframe().iloc[:, 2],
                "forecast_eer": pred.pd_dataframe().iloc[:, 3],
                "forecast_ir": pred.pd_dataframe().iloc[:, 4]
            })

            forecasts[horizon] = pred_df

            output_file = f"italy_forecasts_tftx_{horizon}m.csv"
            pred_df.to_csv(output_file, index=True)
            print(f"Forecast saved to {output_file}")

        return forecasts


def main():
    import sys, darts
    print(sys.version)
    print("torch:", torch.__version__)
    print("darts:", darts.__version__)
    print("lightning:", pl.__version__)
    print("numpy:", np.__version__)
    print("pandas:", pd.__version__)
    print("cuda available:", torch.cuda.is_available())

    generator = italyForecastGenerator(
        data_path="/content/all_mulvar_data_italy_v2.csv",
        random_seed=1
    )

    generator.generate_forecasts([12, 24])

    print("\nCreated files:")
    print("italy_forecasts_tftx_12m.csv")
    print("italy_forecasts_tftx_24m.csv")


if __name__ == "__main__":
    main()

INFO:lightning_fabric.utilities.seed:Seed set to 1
INFO:lightning_fabric.utilities.seed:Seed set to 1


3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
torch: 2.3.1+cu121
darts: 0.34.0
lightning: 2.2.5
numpy: 1.26.4
pandas: 2.2.2
cuda available: True

Generating 12-month forecast...


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/setup.py:187: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
INFO:pytorch_lightning.callbacks.model_summary:
   | Name                              | Type                             | Params
----------------------------------------------------------------------------------------
0  | train_metrics                     | MetricCollection                 | 0     
1  | val_metrics                       | MetricCollection                 | 0     
2  | input_embeddings                  | _MultiEmbedding                  | 0     
3  | static_covariates_vsn             | _Va

Training model for 12-month horizon...


Training: |          | 0/? [00:00<?, ?it/s]

...

Training: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=300` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


Generating 24-month predictions...


Predicting: |          | 0/? [00:00<?, ?it/s]

Forecast saved to italy_forecasts_tftx_24m.csv

Created files:
italy_forecasts_tftx_12m.csv
italy_forecasts_tftx_24m.csv
